# Parameter study for Electrochemistry and Electrolyzer Stack Models

    This notebook regenerates the parameter-study table and figure embedded in the
    chapter. It is intentionally lightweight so PaperLab can refresh the visible
    results while the full companion notebook remains available for broader
    branch-specific NeqSim workflows.


In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path(globals().get("__vsc_ipynb_file__", "expected_output.ipynb")).resolve().parent
FIGURES_DIR = NOTEBOOK_DIR.parent / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

headers = ["Technology", "Cell voltage (V)", "Specific energy (kWh/kg H2)", "Relative CAPEX index"]
rows = [["PEM", 1.8, 52.0, 1.0], ["Alkaline", 1.85, 56.0, 0.75], ["SOEC", 1.3, 38.0, 1.7], ["AEM", 1.85, 60.0, 1.15]]
series = [["Specific energy", [52.0, 56.0, 38.0, 60.0]], ["CAPEX index x40", [40.0, 30.0, 68.0, 46.0]]]
study = {"caption": "Expected electrolyzer technology output from stack I-V and cost-hook cells.", "description": "The notebook output should make the technology trade-off visible: SOEC has the lowest electrical specific energy in this screening basis, while low-temperature technologies carry different pressure, current-density, and cost assumptions.", "discussion": "The parameter study shows how green and pink H2 production from water and power responds when the controlling parameter changes. The important result is not a single base-case value; it is the shape of the response and the point where stack power, specific energy, pressure, temperature, and H2 production begin to trade against margin or cost.", "calculation_basis": "Faraday-law water splitting, cell-voltage specific energy, current-density/load sweeps, and NeqSim Electrolyzer technology defaults where the notebook uses a stack model.", "study_basis": "Public DOE electrolysis technology descriptions, IRENA scale/cost-learning observations, and the Buttler-Spliethoff electrolysis review define the technology bands.", "source_labels": "US DOE electrolysis technology overview, IRENA Green Hydrogen Cost Reduction, Buttler and Spliethoff water-electrolysis review, ISO 22734 electrolyzer safety basis", "citation": "\\cite{doeElectrolysis,irena2022greenhydrogen,buttler2018water,iso22734}"}
topic_examples = [{"route": "Green hydrogen", "number": 1, "topic": "Alkaline water electrolysis", "chapter": "ch07_electrochemistry_and_electrolyzers", "calculation": "Run ElectrolyzerTechnology.ALKALINE with current-density and voltage assumptions from the technology selector.", "sweep": "Current density, operating pressure, Faradaic efficiency, and stack temperature.", "kpis": "Cell voltage, kWh/kg H2, hydrogen rate, cooling or heat-rejection index.", "source_ids": ["doeElectrolysis", "buttler2018water", "irena2022greenhydrogen"], "source_labels": "US DOE electrolysis technology overview, Buttler and Spliethoff water-electrolysis review, IRENA Green Hydrogen Cost Reduction", "citation": "\\cite{doeElectrolysis,buttler2018water,irena2022greenhydrogen}"}, {"route": "Green hydrogen", "number": 2, "topic": "PEM electrolysis", "chapter": "ch07_electrochemistry_and_electrolyzers", "calculation": "Run ElectrolyzerTechnology.PEM with pressure operation, I-V curve, and compression-offset accounting.", "sweep": "Current density, pressure, membrane resistance, and Faradaic efficiency.", "kpis": "Specific energy, stack power, pressure credit, CAPEX index, durability attention index.", "source_ids": ["carmo2013pem", "doeElectrolysis", "iso22734"], "source_labels": "Carmo et al. PEM electrolysis review, US DOE electrolysis technology overview, ISO 22734 electrolyzer safety basis", "citation": "\\cite{carmo2013pem,doeElectrolysis,iso22734}"}, {"route": "Green hydrogen", "number": 3, "topic": "Anion exchange membrane electrolysis", "chapter": "ch07_electrochemistry_and_electrolyzers", "calculation": "Run ElectrolyzerTechnology.AEM as an emerging low-precious-metal case with a membrane-resistance sensitivity.", "sweep": "AEM resistance factor, pressure, current density, and technology maturity factor.", "kpis": "Specific energy, CAPEX index, durability risk index, water-management attention.", "source_ids": ["doeElectrolysis", "irena2022greenhydrogen", "roger2017catalysts"], "source_labels": "US DOE electrolysis technology overview, IRENA Green Hydrogen Cost Reduction, Roger et al. water-splitting catalyst review", "citation": "\\cite{doeElectrolysis,irena2022greenhydrogen,roger2017catalysts}"}, {"route": "Green hydrogen", "number": 4, "topic": "Solid oxide electrolysis cells", "chapter": "ch07_electrochemistry_and_electrolyzers", "calculation": "Run ElectrolyzerTechnology.SOEC with high-temperature efficiency and heat-credit assumptions.", "sweep": "Stack temperature, heat-credit fraction, current density, and steam-feed condition.", "kpis": "Electrical kWh/kg H2, thermal-duty credit, overall energy index, high-temperature materials attention.", "source_ids": ["doeElectrolysis", "buttler2018water", "irena2022greenhydrogen"], "source_labels": "US DOE electrolysis technology overview, Buttler and Spliethoff water-electrolysis review, IRENA Green Hydrogen Cost Reduction", "citation": "\\cite{doeElectrolysis,buttler2018water,irena2022greenhydrogen}"}, {"route": "Green hydrogen", "number": 5, "topic": "Electrocatalysts for HER and OER", "chapter": "ch07_electrochemistry_and_electrolyzers", "calculation": "Represent HER/OER catalyst improvements as reduced activation overpotential in the I-V characteristic.", "sweep": "HER overpotential, OER overpotential, Tafel slope proxy, and noble-metal loading index.", "kpis": "Cell-voltage reduction, kWh/kg H2 reduction, catalyst-cost index, stability attention index.", "source_ids": ["roger2017catalysts", "buttler2018water", "doeElectrolysis"], "source_labels": "Roger et al. water-splitting catalyst review, Buttler and Spliethoff water-electrolysis review, US DOE electrolysis technology overview", "citation": "\\cite{roger2017catalysts,buttler2018water,doeElectrolysis}"}, {"route": "Green hydrogen", "number": 6, "topic": "Membrane and ionomer development", "chapter": "ch07_electrochemistry_and_electrolyzers", "calculation": "Model membrane changes as area-specific resistance, crossover, and degradation factors in a stack screening table.", "sweep": "Membrane resistance, gas-crossover index, PFAS-free material flag, and lifetime multiplier.", "kpis": "Ohmic-loss voltage, Faradaic-efficiency loss, durability index, product-purity attention.", "source_ids": ["carmo2013pem", "iso22734", "doeElectrolysis"], "source_labels": "Carmo et al. PEM electrolysis review, ISO 22734 electrolyzer safety basis, US DOE electrolysis technology overview", "citation": "\\cite{carmo2013pem,iso22734,doeElectrolysis}"}]
df = pd.DataFrame(rows, columns=headers)
display(df)
df.to_csv(FIGURES_DIR / "notebook_electrolyzer_technology_output.csv", index=False)
if topic_examples:
    topic_df = pd.DataFrame(topic_examples)
    display(topic_df[["route", "number", "topic", "calculation", "sweep", "kpis", "source_labels"]])
    topic_df.to_csv(FIGURES_DIR / "notebook_electrolyzer_technology_output_topic_examples.csv", index=False)


In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "figure.dpi": 150,
    "savefig.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.35,
})

x = list(df[headers[0]])
fig, ax = plt.subplots(figsize=(7.0, 4.0))
for name, values in series:
    ax.plot(x, values, marker="o", linewidth=1.8, label=name)
ax.set_xlabel("Technology case")
ax.set_ylabel("Specific energy (kWh/kg H2)")
ax.set_title("Expected electrolyzer technology output from stack I-V and cost-hook cells.")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "notebook_electrolyzer_technology_output.png", dpi=150, bbox_inches="tight")
plt.show()
print("wrote", FIGURES_DIR / "notebook_electrolyzer_technology_output.png")


## Results and discussion

The notebook output should make the technology trade-off visible: SOEC has the lowest electrical specific energy in this screening basis, while low-temperature technologies carry different pressure, current-density, and cost assumptions.

**Calculation basis.** Faraday-law water splitting, cell-voltage specific energy, current-density/load sweeps, and NeqSim Electrolyzer technology defaults where the notebook uses a stack model.

**Study basis.** Public DOE electrolysis technology descriptions, IRENA scale/cost-learning observations, and the Buttler-Spliethoff electrolysis review define the technology bands. \cite{doeElectrolysis,irena2022greenhydrogen,buttler2018water,iso22734}

**Discussion.** The parameter study shows how green and pink H2 production from water and power responds when the controlling parameter changes. The important result is not a single base-case value; it is the shape of the response and the point where stack power, specific energy, pressure, temperature, and H2 production begin to trade against margin or cost.

## Literature-review topic calculation examples

The table below maps the requested blue, green, and frontier hydrogen literature
topics to calculation examples in the chapter where the physics and NeqSim
workflow fit best. Each row names the calculation to run, the input sweep that
makes it useful as a study example, and the KPIs that should appear in the
notebook output or discussion.

| Review list | No. | Topic | Calculation example placed here | Sweep or input basis | Output KPIs | Literature basis |
|---|---:|---|---|---|---|---|
| Green hydrogen | 1 | Alkaline water electrolysis | Run ElectrolyzerTechnology.ALKALINE with current-density and voltage assumptions from the technology selector. | Current density, operating pressure, Faradaic efficiency, and stack temperature. | Cell voltage, kWh/kg H2, hydrogen rate, cooling or heat-rejection index. | US DOE electrolysis technology overview, Buttler and Spliethoff water-electrolysis review, IRENA Green Hydrogen Cost Reduction |
| Green hydrogen | 2 | PEM electrolysis | Run ElectrolyzerTechnology.PEM with pressure operation, I-V curve, and compression-offset accounting. | Current density, pressure, membrane resistance, and Faradaic efficiency. | Specific energy, stack power, pressure credit, CAPEX index, durability attention index. | Carmo et al. PEM electrolysis review, US DOE electrolysis technology overview, ISO 22734 electrolyzer safety basis |
| Green hydrogen | 3 | Anion exchange membrane electrolysis | Run ElectrolyzerTechnology.AEM as an emerging low-precious-metal case with a membrane-resistance sensitivity. | AEM resistance factor, pressure, current density, and technology maturity factor. | Specific energy, CAPEX index, durability risk index, water-management attention. | US DOE electrolysis technology overview, IRENA Green Hydrogen Cost Reduction, Roger et al. water-splitting catalyst review |
| Green hydrogen | 4 | Solid oxide electrolysis cells | Run ElectrolyzerTechnology.SOEC with high-temperature efficiency and heat-credit assumptions. | Stack temperature, heat-credit fraction, current density, and steam-feed condition. | Electrical kWh/kg H2, thermal-duty credit, overall energy index, high-temperature materials attention. | US DOE electrolysis technology overview, Buttler and Spliethoff water-electrolysis review, IRENA Green Hydrogen Cost Reduction |
| Green hydrogen | 5 | Electrocatalysts for HER and OER | Represent HER/OER catalyst improvements as reduced activation overpotential in the I-V characteristic. | HER overpotential, OER overpotential, Tafel slope proxy, and noble-metal loading index. | Cell-voltage reduction, kWh/kg H2 reduction, catalyst-cost index, stability attention index. | Roger et al. water-splitting catalyst review, Buttler and Spliethoff water-electrolysis review, US DOE electrolysis technology overview |
| Green hydrogen | 6 | Membrane and ionomer development | Model membrane changes as area-specific resistance, crossover, and degradation factors in a stack screening table. | Membrane resistance, gas-crossover index, PFAS-free material flag, and lifetime multiplier. | Ohmic-loss voltage, Faradaic-efficiency loss, durability index, product-purity attention. | Carmo et al. PEM electrolysis review, ISO 22734 electrolyzer safety basis, US DOE electrolysis technology overview |



In [ ]:
result = {
    "chapter": "ch07_electrochemistry_and_electrolyzers",
    "title": "Electrochemistry and Electrolyzer Stack Models",
    "figure": "notebook_electrolyzer_technology_output.png",
    "caption": "Expected electrolyzer technology output from stack I-V and cost-hook cells.",
    "description": "The notebook output should make the technology trade-off visible: SOEC has the lowest electrical specific energy in this screening basis, while low-temperature technologies carry different pressure, current-density, and cost assumptions.",
    "discussion": "The parameter study shows how green and pink H2 production from water and power responds when the controlling parameter changes. The important result is not a single base-case value; it is the shape of the response and the point where stack power, specific energy, pressure, temperature, and H2 production begin to trade against margin or cost.",
    "calculation_basis": "Faraday-law water splitting, cell-voltage specific energy, current-density/load sweeps, and NeqSim Electrolyzer technology defaults where the notebook uses a stack model.",
    "study_basis": "Public DOE electrolysis technology descriptions, IRENA scale/cost-learning observations, and the Buttler-Spliethoff electrolysis review define the technology bands.",
    "source_labels": "US DOE electrolysis technology overview, IRENA Green Hydrogen Cost Reduction, Buttler and Spliethoff water-electrolysis review, ISO 22734 electrolyzer safety basis",
    "citation": "\\cite{doeElectrolysis,irena2022greenhydrogen,buttler2018water,iso22734}",
    "topic_examples": topic_examples,
    "rows": df.to_dict(orient="records"),
}
with open(FIGURES_DIR / "notebook_electrolyzer_technology_output_results.json", "w", encoding="utf-8") as handle:
    json.dump(result, handle, indent=2)
result
